# Pipeline de Treinamento DQN para QA com NLPGym
Este notebook mostra passo a passo como preparar, treinar e avaliar um agente DQN para tarefas de Question Answering usando a biblioteca **nlp-gym**.

## 1. Instalar dependências
Execute no terminal para garantir que todas as bibliotecas necessárias estejam instaladas:
```bash
pip install -r requirements.txt
pip install stable-baselines3 torch scikit-learn nbformat
```

## 2. Imports e verificação do ambiente (com wrapper de compatibilidade)

In [ ]:
from gymnasium import Env
from stable_baselines3.common.env_checker import check_env
from nlp_gym.envs.question_answering.env import QAEnv
from nlp_gym.envs.question_answering.featurizer import InformedFeaturizer

# Wrapper de compatibilidade com Gymnasium
class EnvCompatibility(Env):
    def __init__(self, legacy_env):
        super().__init__()
        self.legacy_env = legacy_env
        self.action_space = legacy_env.action_space
        self.observation_space = legacy_env.observation_space

    def reset(self, **kwargs):
        obs = self.legacy_env.reset(**kwargs)
        return obs, {}

    def step(self, action):
        obs, reward, done, info = self.legacy_env.step(action)
        return obs, reward, done, False, info

    def render(self, *args, **kwargs):
        return self.legacy_env.render(*args, **kwargs)

    def close(self):
        return self.legacy_env.close()

# Teste de conformidade
featurizer = InformedFeaturizer()
raw_env = QAEnv(observation_featurizer=featurizer)
compat_env = EnvCompatibility(raw_env)
check_env(compat_env, warn=True)

## 3. Preparar dados e vetorização

In [ ]:
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from nlp_gym.data_pools.custom_question_answering_pools import QASC

# Carrega dados de treino
train_pool = QASC.prepare("train")

# Cria ambiente base e adiciona samples
featurizer = InformedFeaturizer()
base_env = QAEnv(observation_featurizer=featurizer)
for sample, weight in train_pool:
    base_env.add_sample(sample, weight)

# Adiciona wrappers de Monitor e VecEnv
env = Monitor(base_env)
vec_env = DummyVecEnv([lambda: env])

## 4. Configurar modelo DQN com callback de avaliação

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback
from nlp_gym.data_pools.custom_question_answering_pools import QASC

# Preparar ambiente de validação
val_pool = QASC.prepare("val")
eval_base_env = QAEnv(observation_featurizer=featurizer)
for sample, weight in val_pool:
    eval_base_env.add_sample(sample, weight)
eval_env = Monitor(eval_base_env)
eval_vec_env = DummyVecEnv([lambda: eval_env])

# Callback para avaliação periódica e salvamento do melhor modelo
eval_callback = EvalCallback(
    eval_vec_env,
    best_model_save_path="models/",
    log_path="logs/",
    eval_freq=1000,
    deterministic=True,
    render=False
)

# Instanciar modelo DQN
model = DQN(
    policy="MlpPolicy",
    env=vec_env,
    gamma=0.99,
    batch_size=32,
    learning_rate=1e-4,
    exploration_fraction=0.1,
    policy_kwargs={"net_arch": [64, 64]},
    verbose=1
)

## 5. Treinar o agente

In [ ]:
import os

# Cria diretório para salvar modelos e logs
os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Treinar
model.learn(total_timesteps=int(1e5), callback=eval_callback)

## 6. Carregar melhor modelo e plotar curva de aprendizado

In [ ]:
import matplotlib.pyplot as plt
from stable_baselines3 import DQN

# Carregar melhor modelo
best_model = DQN.load("models/best_model.zip", env=vec_env)

# Plot curva de recompensas e comprimentos
if hasattr(model, 'ep_info_buffer') and len(model.ep_info_buffer) > 0:
    rewards = [ep['r'] for ep in model.ep_info_buffer]
    lengths = [ep['l'] for ep in model.ep_info_buffer]

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(rewards)
    plt.title('Recompensa por Episódio')
    plt.subplot(1, 2, 2)
    plt.plot(lengths)
    plt.title('Comprimento por Episódio')
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado de progresso disponível.")

## 7. Avaliar no conjunto de validação

In [ ]:
from tqdm import tqdm

num_episodes = 100
correct = 0
for _ in tqdm(range(num_episodes), desc='Avaliação'):
    obs = eval_vec_env.reset()
    done = False
    ep_reward = 0
    while not done:
        action, _ = best_model.predict(obs, deterministic=True)
        obs, reward, done, info = eval_vec_env.step(action)
        ep_reward += reward
    if ep_reward > 0:
        correct += 1
print(f"Acurácia final em {num_episodes} episódios: {correct/num_episodes:.2f}")

## Próximos passos
1. Ajustar `total_timesteps` e hiperparâmetros conforme desempenho observado.
2. Executar:
   ```bash
   jupyter notebook /mnt/data/qa_training_pipeline.ipynb
   ```
3. Explorar os logs em `logs/` e modelos em `models/`.
4. Métricas adicionais: medir tempo por episódio, complexidade do featurizer, etc.